# Whole-frame deployment benchmark

Profile one backend through the fixed inference API. Real loading/profiling and CPU fixtures are separate explicit actions below; both default off. See [benchmark usage and metric definitions](../docs/benchmark.md). Reference annotations never enter backend inputs.


In [ ]:
REPO_ROOT_OVERRIDE = None
DATA_ROOT = None
RUN_ROOT = None
CACHE_ROOT = None

RUN_NAME = "rellis_material_v1"
MODEL_KEY = "clip_vit_b32"  # qwen3_vl_4b / qwen3_5_4b; fallback qwen3_5_2b is explicit.
DEVICE = "cuda:0"  # One explicitly selected visible CUDA device.
MODEL_CACHE_PATH = None  # Defaults to settings cache_dir or CACHE_ROOT/huggingface/hub.
CLIP_CALIBRATION_PATH = None  # Defaults to metadata/clip_calibration.json within RUN_NAME.
BENCHMARK_CONFIG_PATH = None  # Defaults to configs/benchmark/default.json.
INFERENCE_SETTINGS_PATH = None  # JSON settings dictionary, or {"settings": {...}}.
RUN_BENCHMARK = False
RUN_FAKE_BENCHMARK = False


In [ ]:
import json
import os
from pathlib import Path

configured_repo = REPO_ROOT_OVERRIDE or os.environ.get("TRAVERSABILITY_REPO_ROOT")
if configured_repo:
    candidates = [Path(configured_repo).expanduser().resolve()]
else:
    working_dir = Path.cwd().resolve()
    candidates = [working_dir, *working_dir.parents]
candidates = [path for base in candidates for path in (base, base / "VLM_evaluation")]
REPO_ROOT = next(
    (candidate for candidate in candidates
     if (candidate / "pyproject.toml").is_file()
     and (candidate / "docs" / "interfaces.md").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from VLM_evaluation or its checkout, or set REPO_ROOT_OVERRIDE to VLM_evaluation.")

try:
    from traversability_benchmark import resolve_roots
except ModuleNotFoundError as error:
    raise RuntimeError(
        "Install the editable package with scripts/setup_env.py and select its registered kernel. "
        "See docs/setup.md."
    ) from error
roots = resolve_roots(
    REPO_ROOT, data_root=DATA_ROOT, run_root=RUN_ROOT, cache_root=CACHE_ROOT
)

from uuid import uuid4
from traversability_benchmark import freeze_selection, profile_backend

config_path = (
    Path(BENCHMARK_CONFIG_PATH).expanduser()
    if BENCHMARK_CONFIG_PATH else REPO_ROOT / "configs" / "benchmark" / "default.json"
)
if not config_path.is_absolute():
    config_path = REPO_ROOT / config_path
config = json.loads(config_path.read_text(encoding="utf-8"))
for key, expected in {
    "warmup_frames": 5, "test_frames": 20, "measured_repeats": 2,
    "component_budget_bytes": 6_000_000_000,
}.items():
    if config.get(key) != expected:
        raise ValueError(f"{key} must remain {expected} for the fixed benchmark protocol.")

settings = dict(config["settings"])
if INFERENCE_SETTINGS_PATH:
    settings_path = Path(INFERENCE_SETTINGS_PATH).expanduser()
    if not settings_path.is_absolute():
        settings_path = REPO_ROOT / settings_path
    supplied_settings = json.loads(settings_path.read_text(encoding="utf-8"))
    settings = dict(supplied_settings.get("settings", supplied_settings))

run_dir = roots.run_root / RUN_NAME
print(json.dumps({
    "data_root": str(roots.data_root), "run_dir": str(run_dir),
    "cache_root": str(roots.cache_root), "model_key": MODEL_KEY,
    "real_enabled": RUN_BENCHMARK, "fixture_enabled": RUN_FAKE_BENCHMARK,
}, indent=2))
if RUN_BENCHMARK and RUN_FAKE_BENCHMARK:
    raise ValueError("Enable one action at a time.")


## Optional CPU fixture

Set only `RUN_FAKE_BENCHMARK = True` and rerun the configuration/setup cells before this cell. The package creates marked synthetic fixtures beneath distinct run/data roots and injects a fake backend. These are software-test artifacts, never model experiment results.


In [ ]:
if RUN_FAKE_BENCHMARK:
    from traversability_benchmark import FakeBackend, NullMemoryProbe, prepare_fixture

    fixture_id = "deployment_fixture_" + uuid4().hex
    fixture_run_dir = roots.run_root / "fixtures" / fixture_id
    fixture_data_root = roots.data_root / "fixtures" / fixture_id
    fixture_selection = prepare_fixture(fixture_run_dir, fixture_data_root)
    fixture_profile_dir = fixture_run_dir / "benchmark" / "fixture_cpu" / uuid4().hex
    fixture_result = profile_backend(
        "fixture_cpu", settings, fixture_selection,
        data_root=fixture_data_root, output_dir=fixture_profile_dir,
        backend_factory=FakeBackend, memory_probe=NullMemoryProbe(), fixture=True,
    )
    print(json.dumps({
        "artifact_dir": str(fixture_profile_dir),
        "fixture": True, "summary": fixture_result["summary"],
    }, indent=2, allow_nan=False))
else:
    print("CPU fixture disabled. Set RUN_FAKE_BENCHMARK = True to opt in.")


## Optional real profile

Prepare real RGB/SAM manifests and explicitly cache the checkpoint using the inference workflow first. Set only `RUN_BENCHMARK = True`. This cell forces offline checkpoint loading, freezes/reuses five development and 20 test frames, and runs two measured repeats. It loads one backend and closes that backend in the profiler.

Use the exact frozen inference settings, including CLIP's development-calibrated rejection threshold where applicable. Actual metadata comes from the loaded backend and is compared with `metadata/<model_key>.json` when present; missing/incomplete metadata is recorded rather than inferred from requested settings. Real loading refuses measurable nonzero allocated/reserved Torch baselines: use a dedicated idle kernel. Other modules may remain resident in separate GPU processes. Peer adapters currently clear unused process allocator caches during close; shared-kernel cleanup integration remains pending (see docs/benchmark.md).


In [ ]:
if RUN_BENCHMARK:
    os.environ["TRAVERSABILITY_DATA_ROOT"] = str(roots.data_root)
    os.environ["TRAVERSABILITY_RUN_ROOT"] = str(roots.run_root)
    os.environ["TRAVERSABILITY_CACHE_ROOT"] = str(roots.cache_root)
    os.environ["HF_HOME"] = str(roots.cache_root / "huggingface")
    os.environ["HF_HUB_CACHE"] = str(roots.cache_root / "huggingface" / "hub")
    os.environ["TORCH_HOME"] = str(roots.cache_root / "torch")
    os.environ["XDG_CACHE_HOME"] = str(roots.cache_root)
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"

    real_settings = dict(settings)
    real_settings["device"] = DEVICE
    configured_model_cache = MODEL_CACHE_PATH or real_settings.get("cache_dir")
    model_cache = (
        Path(configured_model_cache).expanduser()
        if configured_model_cache else roots.cache_root / "huggingface" / "hub"
    )
    if not model_cache.is_absolute():
        model_cache = roots.cache_root / model_cache
    real_settings["cache_dir"] = str(model_cache)
    os.environ["HF_HUB_CACHE"] = str(model_cache)
    real_settings["local_files_only"] = True
    if MODEL_KEY == "clip_vit_b32":
        configured_calibration = CLIP_CALIBRATION_PATH or real_settings.get("calibration_path")
        calibration_path = (
            Path(configured_calibration).expanduser()
            if configured_calibration else run_dir / "metadata" / "clip_calibration.json"
        )
        if not calibration_path.is_absolute():
            calibration_path = run_dir / calibration_path
        if not calibration_path.is_file():
            raise RuntimeError(
                "CLIP needs development-set calibration first. Run the inference calibration "
                "workflow, or set CLIP_CALIBRATION_PATH to its compatible JSON file."
            )
        real_settings["calibration_path"] = str(calibration_path)

    selection = freeze_selection(
        run_dir, roots.data_root, seed=config["selection_seed"]
    )
    metadata_path = run_dir / "metadata" / f"{MODEL_KEY}.json"
    inference_metadata = (
        json.loads(metadata_path.read_text(encoding="utf-8"))
        if metadata_path.is_file() else None
    )
    print(json.dumps({
        "frozen_selection_fingerprint": selection["fingerprint"],
        "coverage": selection["coverage"],
        "inference_metadata_sidecar_present": inference_metadata is not None,
    }, indent=2, allow_nan=False))
    profile_dir = run_dir / "benchmark" / MODEL_KEY / uuid4().hex
    real_result = profile_backend(
        MODEL_KEY, real_settings, selection, data_root=roots.data_root,
        output_dir=profile_dir, inference_metadata=inference_metadata,
    )
    print(json.dumps({
        "artifact_dir": str(profile_dir),
        "summary": real_result["summary"],
        "comparison_metadata_complete": real_result["metadata"]["comparison_metadata_complete"],
        "missing_comparison_metadata": real_result["metadata"]["missing_comparison_metadata"],
        "inference_metadata_mismatches": real_result["metadata"]["inference_metadata_mismatches"],
    }, indent=2, allow_nan=False))
else:
    print("Real benchmark disabled. Set RUN_BENCHMARK = True to opt in.")


Loading and the first-call/cold-start measurement are separate from 40 warmed frame observations. All preparation, requests and parsing inside `predict_frame` enter wall-clock time. Failed calls retain their time and are reported alongside successful-only statistics.

Allocator peaks and device-wide sampled usage are distinct; missing GPU measurements are `null`. Component increments use the pre-load resident baseline so weights count; telemetry/CUDA context initialization overhead is excluded and recorded. Fixture budget verdicts remain null even with injected statistics. A sampled device peak is a lower bound and includes other processes. The 6 GB component target and the final full-pipeline budget require real GPU validation under active component load.

Profiles preserve IDs/timestamps and source coverage. The default public RELLIS benchmark needs no phone clips or bulk manual labels. Its material-policy references and held-out campus recordings do not establish cup/cable accuracy, independent geography or physical traversability.
